# Semana 13 — Alertas industriais com estado e ciclo de vida

**Roteiro:** revisão expressa da Semana 12 (QoS, falhas e duplicidade); depois, foco em janelas, histerese, persistência, severidade, reconhecimento e normalização. Cenário: prensa `prensa-01` envia temperatura por MQTT ao FactoryHub.

## Objetivos

- explicar por que QoS 1 pode repetir mensagens e por que alertas precisam de estado;
- distinguir janela, persistência, histerese e supressão;
- modelar abertura, reconhecimento e normalização;
- validar uma regra com sequência de leituras.

## Revisão rápida da Semana 12 — 8 minutos

1. O que QoS 1 garante? Pode haver duplicidade?
2. Qual campo identifica uma retransmissão?
3. Mensagem retida é histórico?
4. Para que servem `catch` e `status` no Node-RED?

**Gabarito:** QoS 1 busca entrega ao menos uma vez e pode repetir; deduplique por `event_id`; mensagem retida preserva o último estado do tópico, não uma série histórica; `catch` captura erros e `status` expõe o estado de nós/conexão.

## Da leitura ao alerta

Regra simples (`temperatura > 75`) pode notificar a cada mensagem e oscilar junto com o sensor. Exemplo didático: abrir após **3 leituras altas** em sequência; manter o alerta aberto até a temperatura cair a **70 °C ou menos**. A faixa entre 70 e 75 °C é histerese.

Estados possíveis: `NORMAL → PENDENTE → ABERTO → RECONHECIDO → NORMALIZADO`. Reconhecer significa que alguém viu o alerta; não significa que a máquina voltou ao normal.

## Exemplo — histerese e persistência

| Leitura | Regra única >75 | Regra com histerese |
|---:|---|---|
| 75,2 | alerta | conta 1 alta |
| 74,9 | normal | conta volta a zero antes da abertura |
| 76,0; 77,0; 78,0 | três alertas | abre um alerta após a terceira |
| 74,0 | normal | mantém alerta aberto |
| 69,5 | normal | normaliza |

Persistência reduz ruído, mas atrasa a detecção. Ajuste a regra conforme risco, frequência de amostragem e validação com a equipe da planta.

## Contrato da telemetria

```json
{"event_id":"evt-42","equipment_id":"prensa-01","timestamp":"2026-10-27T17:05:00Z","temperature_c":78.4}
```

Use instante com fuso/UTC, unidade explícita e identificador único. Não converta payload inválido para zero: isso pode mascarar uma falha de sensor.

In [ ]:
limite_abertura = 75.0
limite_normalizacao = 70.0
leituras_para_abrir = 3
estado = {"status": "NORMAL", "altas": 0, "alerta_id": None}

for i, temp in enumerate([76.1, 77.0, 78.4, 74.9, 72.0, 69.5], start=1):
    if estado["status"] == "NORMAL":
        estado["altas"] = estado["altas"] + 1 if temp >= limite_abertura else 0
        if estado["altas"] >= leituras_para_abrir:
            estado.update(status="ABERTO", alerta_id=f"ALT-prensa-01-{i:03d}")
            print(i, temp, "ABRIR", estado["alerta_id"])
        else:
            print(i, temp, "NORMAL / persistência", estado["altas"])
    elif temp <= limite_normalizacao:
        print(i, temp, "NORMALIZAR", estado["alerta_id"])
        estado.update(status="NORMAL", altas=0, alerta_id=None)
    else:
        print(i, temp, estado["status"], "manter alerta")

estado

A terceira alta abre o alerta. Leituras entre 70 °C e 75 °C não o fecham; apenas a queda a 70 °C ou menos normaliza.

## Reconhecimento, severidade e supressão

- **Aberto:** condição detectada e ainda ativa.
- **Reconhecido:** operador confirmou ciência; a condição pode persistir.
- **Normalizado:** condição deixou de existir pela regra técnica.

Guarde `alerta_id`, abertura, reconhecimento (quem/quando), severidade e normalização. Suprima notificações repetidas para o mesmo alerta aberto, mas não oculte escalada de severidade nem recorrência após normalização.

In [ ]:
# Reconhecer não altera a condição física nem apaga evidências.
if estado["status"] == "ABERTO":
    estado["status"] = "RECONHECIDO"
    estado["reconhecido_por"] = "operador.turno_a"
estado

## Do MQTT ao painel Node-RED

```text
mqtt in → json → validar → deduplicar event_id → atualizar estado por equipamento
        → classificar/normalizar → persistir transição → painel/notificação
```

Para várias instâncias, estado em memória de um Function node não basta: use armazenamento compartilhado e operações idempotentes. Inclua rota de erro e observabilidade.

## Atividade — tabela de transições

Com abertura ≥75 °C, retorno ≤70 °C e três altas, analise:

`74, 76, 77, 78, 76, 74, 71, 69, 77, 78, 79`

Para cada trecho, registre estado antes/depois e se deve notificar. Depois simule o mesmo `event_id` duas vezes e explique por que a duplicata não pode contar como duas leituras. Acrescente uma regra crítica a 90 °C sem remover a histerese.

## Entrega e referências

Demonstre condição normal, abertura por persistência, reconhecimento sem normalização, normalização, recorrência e deduplicação. Entregue regra e evidência do fluxo/implementação.

- OASIS. MQTT 3.1.1: <https://docs.oasis-open.org/mqtt/mqtt/v3.1.1/os/mqtt-v3.1.1-os.html>
- Node-RED, mensagens: <https://nodered.org/docs/user-guide/messages>
- Node-RED, Function: <https://nodered.org/docs/user-guide/writing-functions>
- NIST SP 800-82 Rev. 3: <https://csrc.nist.gov/pubs/sp/800/82/r3/final>